In [1]:
import os
import shutil

In [2]:
# 1. Install Library
# !pip install nnunetv2 > /dev/null
!pip install nnunetv2 hiddenlayer graphviz --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.6/205.6 kB 5.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 127.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 148.8 MB/s eta 0:00:

## 1. Giải nén `nnUNet_ra`w & `nnUNet_preprocessed`

In [3]:
import os
import zipfile
from tqdm import tqdm

RAW_ZIP = "/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_raw/Dataset101_BraTS2020.zip"
PRE_ZIP = "/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_preprocessed/Dataset101_BraTS2020.zip"

RAW_DIR = "/content"
PRE_DIR = "/content"

def unzip(zip_path, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        for f in tqdm(z.namelist()):
            z.extract(f, out_dir)

In [4]:
unzip(RAW_ZIP, RAW_DIR)

100%|██████████| 1848/1848 [01:58<00:00, 15.63it/s]


In [5]:
unzip(PRE_ZIP, PRE_DIR)

100%|██████████| 2590/2590 [03:36<00:00, 11.95it/s]


## 2. Set biến môi trường nnU-Net v2

In [6]:
import os

os.environ["nnUNet_raw"] = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results"

print("nnUNet_raw:", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed:", os.environ["nnUNet_preprocessed"])
print("nnUNet_results:", os.environ["nnUNet_results"])

nnUNet_raw: /content/nnUNet_raw
nnUNet_preprocessed: /content/nnUNet_preprocessed
nnUNet_results: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results


## 3. Check split file

In [7]:
!cp /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_preprocessed/Dataset101_BraTS2020/splits_final.json \
/content/nnUNet_preprocessed/Dataset101_BraTS2020/splits_final.json

In [8]:
import json

split_path = "/content/nnUNet_preprocessed/Dataset101_BraTS2020/splits_final.json"

with open(split_path) as f:
    splits = json.load(f)

print(f"Số folds: {len(splits)}")
print("Fold 0:", splits[0].keys())
print("Val size fold 0:", len(splits[0]["val"]))

Số folds: 5
Fold 0: dict_keys(['train', 'val'])
Val size fold 0: 59


# B. Train EDL nnU-Net v2 (3D fullres)

In [9]:
# Inject Custom Trainer
import os
import sys
import site
import shutil
import nnunetv2 # Import để tìm đường dẫn cài đặt

# Đường dẫn file gốc của bạn trên Drive
source_trainer = '/content/drive/MyDrive/NCKH/nnUnet/src/trainers/EDLTrainer.py'

# Tìm đường dẫn thư viện nnU-Net v2 trong môi trường Colab
install_path = os.path.dirname(nnunetv2.__file__) # /usr/local/lib/python3.10/dist-packages/nnunetv2
print(install_path)

target_folder = os.path.join(install_path, "training", "nnUNetTrainer")
target_file = os.path.join(target_folder, "EDLTrainer.py")

# Copy file
if os.path.exists(source_trainer):
    shutil.copy(source_trainer, target_file)
    print(f"Đã tiêm EDLTrainer vào: {target_file}")
else:
    print("Lỗi: Không tìm thấy file EDLTrainer trên Drive!")

/usr/local/lib/python3.12/dist-packages/nnunetv2
Đã tiêm EDLTrainer vào: /usr/local/lib/python3.12/dist-packages/nnunetv2/training/nnUNetTrainer/EDLTrainer.py


## 1. Fold 0 1 2 3 4 5

In [10]:
%env nnUNet_n_proc_DA=8

# 2. Chạy 5 Fold đồng thời
# 4 Fold đầu chạy ngầm (Background)
!nohup nnUNetv2_train Dataset101_BraTS2020 3d_fullres 0 -tr EDLTrainer_50epochs_FixedSplit --npz --c > fold0.log 2>&1 &
!nohup nnUNetv2_train Dataset101_BraTS2020 3d_fullres 1 -tr EDLTrainer_50epochs_FixedSplit --npz --c > fold1.log 2>&1 &
!nohup nnUNetv2_train Dataset101_BraTS2020 3d_fullres 2 -tr EDLTrainer_50epochs_FixedSplit --npz --c > fold2.log 2>&1 &
!nohup nnUNetv2_train Dataset101_BraTS2020 3d_fullres 3 -tr EDLTrainer_50epochs_FixedSplit --npz --c > fold3.log 2>&1 &

# Fold thứ 5 chạy ở "bề nổi" (Foreground) - KHÔNG có dấu & ở cuối
# Đây là "mỏ neo" giữ cho Cell này luôn ở trạng thái "Busy"
print("Đang khởi động 5 Fold... Fold 4 sẽ in log trực tiếp tại đây.")
!nnUNetv2_train Dataset101_BraTS2020 3d_fullres 4 -tr EDLTrainer_50epochs_FixedSplit --npz --c

env: nnUNet_n_proc_DA=8
Đang khởi động 5 Fold... Fold 4 sẽ in log trực tiếp tại đây.

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-04-14 13:11:34.076414: Using torch.compile...
2026-04-14 13:11:35.469217: do_dummy_2d_data_aug: False
2026-04-14 13:11:35.473231: Using splits from existing split file: /content/nnUNet_preprocessed/Dataset1

In [ ]:
# Giết sạch tất cả các tiến trình liên quan đến nnU-Net
# !pkill -9 -f nnUNetv2_train

In [ ]:
!nvidia-smi

Mon Mar 16 05:29:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   61C    P0             57W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 5. Find best model

In [ ]:
# !nnUNetv2_find_best_configuration Dataset101_BraTS2020 -f 0 1 2 3 4 -tr EDLTrainer -c 3d_fullres


***All results:***
EDLTrainer__nnUNetPlans__3d_fullres: 0.7468481428718251

*Best*: EDLTrainer__nnUNetPlans__3d_fullres: 0.7468481428718251

***Determining postprocessing for best model/ensemble***
Removing all but the largest foreground region did not improve results!
Removing all but the largest component for 1 did not improve results! Dice before: 0.68871 after: 0.66925
Removing all but the largest component for 2 did not improve results! Dice before: 0.79814 after: 0.79008
Removing all but the largest component for 3 did not improve results! Dice before: 0.75369 after: 0.74067

***Run inference like this:***

nnUNetv2_predict -d Dataset101_BraTS2020 -i INPUT_FOLDER -o OUTPUT_FOLDER -f  0 1 2 3 4 -tr EDLTrainer -c 3d_fullres -p nnUNetPlans

***Once inference is completed, run postprocessing like this:***

nnUNetv2_apply_postprocessing -i OUTPUT_FOLDER -o OUTPUT_FOLDER_PP -pp_pkl_file /content/nnUNet_results/Dataset101_BraTS2020/EDLTrainer__nnUNetPlans__3d_fullres/crossval_results_f

## 6. Tính nhãn WT (Whole) TC (Core) ET (Enhancing)

In [ ]:
# import os
# import numpy as np
# import nibabel as nib
# import pandas as pd
# import json
# from multiprocessing import Pool

# # ================= CẤU HÌNH =================
# # Đường dẫn folder chứa nhãn gốc (Ground Truth) - labelsTr
# GT_FOLDER = "/content/nnUNet_raw/Dataset101_BraTS2020/labelsTr"

# # Đường dẫn folder chứa kết quả dự đoán của nnU-Net (Validation Results)
# # Lưu ý: Tìm folder validation_raw hoặc tương tự trong kết quả train
# # Thường là: .../nnUNetTrainer_50epochs__nnUNetPlans__3d_fullres/fold_0/validation
# # NHƯNG vì bạn đã chạy find_best_configuration, nó đã gộp kết quả 5 fold vào folder này:
# PRED_FOLDER = "/content/nnUNet_results/Dataset101_BraTS2020/EDLTrainer__nnUNetPlans__3d_fullres/crossval_results_folds_0_1_2_3_4"

# # OUTPUT_FILE = "BraTS_Final_Metrics.csv"
# OUTPUT_FILE = os.path.join(PRED_FOLDER, "BraTS_EDL_Metrics.csv")

# # ================= HÀM TÍNH TOÁN =================

# def compute_dice(pred_mask, gt_mask):
#     """Tính Dice Score cho Binary Mask"""
#     # Dice = 2 * (A giao B) / (|A| + |B|)
#     intersection = np.sum(pred_mask * gt_mask)
#     sum_pixels = np.sum(pred_mask) + np.sum(gt_mask)

#     if sum_pixels == 0:
#         return 1.0 # Cả 2 đều không có gì -> Đúng tuyệt đối
#     return 2.0 * intersection / sum_pixels

# def process_case(case_id):
#     try:
#         # 1. Đọc file
#         # Tên file nnU-Net thường giữ nguyên hoặc thêm đuôi, check lại format tên file
#         # Ví dụ: BraTS20_Training_001.nii.gz

#         gt_path = os.path.join(GT_FOLDER, f"{case_id}.nii.gz")
#         pred_path = os.path.join(PRED_FOLDER, f"{case_id}.nii.gz")

#         if not os.path.exists(gt_path) or not os.path.exists(pred_path):
#             return None

#         gt_nii = nib.load(gt_path)
#         pred_nii = nib.load(pred_path)

#         gt_data = gt_nii.get_fdata().astype(np.uint8)
#         pred_data = pred_nii.get_fdata().astype(np.uint8)

#         # 2. Định nghĩa các vùng (Regions) theo BraTS 2020
#         # Label 1: NCR/NET
#         # Label 2: ED
#         # Label 3: ET

#         # --- ET (Enhancing Tumor): Label 3 ---
#         gt_ET = (gt_data == 3)
#         pred_ET = (pred_data == 3)

#         # --- TC (Tumor Core): Label 1 + 3 ---
#         gt_TC = np.logical_or(gt_data == 1, gt_data == 3)
#         pred_TC = np.logical_or(pred_data == 1, pred_data == 3)

#         # --- WT (Whole Tumor): Label 1 + 2 + 3 ---
#         gt_WT = (gt_data > 0) # Mọi cái khác 0 đều là u
#         pred_WT = (pred_data > 0)

#         # 3. Tính Dice
#         dice_ET = compute_dice(pred_ET, gt_ET)
#         dice_TC = compute_dice(pred_TC, gt_TC)
#         dice_WT = compute_dice(pred_WT, gt_WT)

#         return {
#             "Case_ID": case_id,
#             "Dice_ET": dice_ET,
#             "Dice_TC": dice_TC,
#             "Dice_WT": dice_WT
#         }
#     except Exception as e:
#         print(f"Error processing {case_id}: {e}")
#         return None

# # ================= MAIN =================
# if __name__ == "__main__":
#     print("Đang tính toán metric BraTS (WT, TC, ET)...")

#     # Lấy danh sách file từ folder dự đoán (chỉ lấy file .nii.gz)
#     files = [f for f in os.listdir(PRED_FOLDER) if f.endswith('.nii.gz')]
#     case_ids = [f.replace('.nii.gz', '') for f in files]

#     print(f"Dataset size: {len(case_ids)} cases")

#     results = []
#     # Chạy loop thường hoặc multiprocessing (Colab có 2 core thì chạy loop cũng nhanh)
#     for i, cid in enumerate(case_ids):
#         res = process_case(cid)
#         if res:
#             results.append(res)
#         if i % 20 == 0:
#             print(f"Processed {i}/{len(case_ids)}")

#     # Tạo DataFrame
#     df = pd.DataFrame(results)

#     # Tính trung bình
#     summary = {
#         "Mean_Dice_ET": df["Dice_ET"].mean(),
#         "Mean_Dice_TC": df["Dice_TC"].mean(),
#         "Mean_Dice_WT": df["Dice_WT"].mean()
#     }

#     print("\n" + "="*30)
#     print("KẾT QUẢ CUỐI CÙNG (EDL Engine)")
#     print("="*30)
#     print(df.describe())
#     print("-" * 30)
#     print(f"Mean Dice WT: {summary['Mean_Dice_WT']:.4f}")
#     print(f"Mean Dice TC: {summary['Mean_Dice_TC']:.4f}")
#     print(f"Mean Dice ET: {summary['Mean_Dice_ET']:.4f}")

#     # Lưu file
#     df.to_csv(OUTPUT_FILE, index=False)
#     print(f"\nĐã lưu kết quả chi tiết vào: {OUTPUT_FILE}")

Đang tính toán metric BraTS (WT, TC, ET)...
Dataset size: 369 cases
Processed 0/369
Processed 20/369
Processed 40/369
Processed 60/369
Processed 80/369
Processed 100/369
Processed 120/369
Processed 140/369
Processed 160/369
Processed 180/369
Processed 200/369
Processed 220/369
Processed 240/369
Processed 260/369
Processed 280/369
Processed 300/369
Processed 320/369
Processed 340/369
Processed 360/369

KẾT QUẢ CUỐI CÙNG (EDL Engine)
          Dice_ET     Dice_TC     Dice_WT
count  369.000000  369.000000  369.000000
mean     0.755689    0.854560    0.914497
std      0.270778    0.178152    0.070873
min      0.000000    0.000000    0.283338
25%      0.750980    0.842993    0.901803
50%      0.859551    0.921038    0.934474
75%      0.913360    0.953626    0.956548
max      1.000000    0.982330    0.982586
------------------------------
Mean Dice WT: 0.9145
Mean Dice TC: 0.8546
Mean Dice ET: 0.7557

Đã lưu kết quả chi tiết vào: /content/nnUNet_results/Dataset101_BraTS2020/EDLTrainer__nnUNe